# LeetCode 4101 — Find All Numbers Disappeared in an Array III (Medium)

Bài toán LeetCode:

```python
class Solution:
    def findDisappearedNumbers(self, nums: List[int], lower: int, upper: int) -> List[List[int]]:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4101 (trên LeetCode bài này thuộc nhóm Premium).

Notebook này dạy bạn kỹ thuật **"chỉ nhìn vào khoảng trống giữa các điểm"**: khi miền giá trị rất lớn (tới `10^9`) nhưng số phần tử ít (tới `10^5`), ta không đi dọc từng số trên trục số mà chỉ xét **khoảng hở giữa các số đã sắp xếp**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì: "số bị thiếu", "đoạn liên tiếp", "gộp đoạn".
2. Vì sao cách đi dọc trục số (Brute Force) đúng nhưng quá chậm khi `upper` tới `10^9`.
3. Công cụ Python cần dùng: `set`, `sorted` / `.sort()`, lọc phần tử, duyệt cặp kề nhau bằng `zip(a, a[1:])`.
4. Ý tưởng **khoảng trống giữa hai số liên tiếp** và **mẹo lính canh (sentinel)**.
5. Cách quét một lượt với biến `next_missing` — gọn, ít bộ nhớ.
6. Cách xử lý số trùng lặp, số nằm ngoài `[lower, upper]`, các trường hợp biên.
7. Phân tích độ phức tạp, lỗi hay gặp, mẹo nhận diện và bài tập vận dụng.

# 1. Đề bài yêu cầu gì?

Cho:

- một mảng số nguyên `nums` (có thể **chưa sắp xếp**, có thể **trùng lặp**),
- hai số nguyên `lower` và `upper`.

Một số nguyên `x` được gọi là **bị thiếu** nếu:

```text
lower <= x <= upper   và   x không xuất hiện trong nums
```

Yêu cầu: trả về **tất cả các số bị thiếu**, nhưng không liệt kê từng số mà **gộp các số thiếu liên tiếp thành đoạn** `[start, end]`.

- Dùng **ít đoạn nhất có thể** (tức là gộp tối đa các số liên tiếp).
- Các đoạn được sắp xếp theo `start` tăng dần.
- Nếu không thiếu số nào, trả về `[]`.

---

## Ví dụ 1

```python
nums = [3, 9, 7], lower = 1, upper = 12
```

Kết quả:

```python
[[1, 2], [4, 6], [8, 8], [10, 12]]
```

Giải thích: trong đoạn `1..12`, các số có mặt là `3, 7, 9`. Các số còn lại bị thiếu và được gộp thành đoạn:

| Đoạn thiếu | Các số trong đoạn |
|---|---|
| `[1, 2]` | 1, 2 |
| `[4, 6]` | 4, 5, 6 |
| `[8, 8]` | 8 (đoạn chỉ có 1 số thì `start == end`) |
| `[10, 12]` | 10, 11, 12 |

## Ví dụ 2

```python
nums = [1, 1, 1000000000], lower = 5, upper = 1000000000
```

Kết quả:

```python
[[5, 999999999]]
```

Số `1` nằm **ngoài** đoạn `[5, 10^9]` nên không ảnh hưởng gì. Số `10^9` có mặt nên đoạn thiếu dừng ở `999999999`.

## Ví dụ 3

```python
nums = [2, 3, 5], lower = 2, upper = 3
```

Kết quả: `[]` vì cả `2` và `3` đều có mặt.

---

## Ràng buộc (constraints)

```text
1 <= nums.length <= 10^5
1 <= nums[i] <= 10^9
1 <= lower <= upper <= 10^9
```

## Nhìn ví dụ 1 trên trục số

Trước khi nghĩ thuật toán, hãy **nhìn bằng mắt**. Cell dưới in từng số từ `lower` tới `upper` và đánh dấu số đó có mặt (`có`) hay bị thiếu (`THIẾU`).

In [1]:
nums = [3, 9, 7]
lower, upper = 1, 12

present = set(nums)
for x in range(lower, upper + 1):
    status = "có" if x in present else "THIẾU"
    print(f"{x:>2}: {status}")

 1: THIẾU
 2: THIẾU
 3: có
 4: THIẾU
 5: THIẾU
 6: THIẾU
 7: có
 8: THIẾU
 9: có
10: THIẾU
11: THIẾU
12: THIẾU


Nhìn cột `THIẾU` ta thấy ngay các cụm liên tiếp: `1-2`, `4-6`, `8`, `10-12`. Đó chính là đáp án.

---

## Đọc kỹ ràng buộc trước khi code

Bài này có **hai loại "kích thước"** khác nhau, phải tách bạch ngay từ đầu:

| Ký hiệu | Ý nghĩa | Lớn nhất |
|---|---|---:|
| `n` | số phần tử của `nums` | `10^5` |
| `L` | độ dài đoạn cần xét, `L = upper - lower + 1` | gần `10^9` |

Cách vừa làm ở trên (đi qua **từng số** từ `lower` tới `upper`) tốn khoảng `L` bước. Với `L ≈ 10^9`, Python cần cỡ vài phút → chắc chắn **Time Limit Exceeded**.

> Câu hỏi then chốt của cả bài: **làm sao trả lời mà chỉ tốn công theo `n`, không theo `L`?**

# 2. Kiến thức Python nền tảng

Trước khi vào lời giải, ta ôn 4 công cụ sẽ dùng.

## 2.1. `set` — tập hợp

`set` lưu các phần tử **không trùng nhau** và kiểm tra `x in s` rất nhanh (trung bình `O(1)`).

- `set(nums)` tự động **bỏ trùng**.
- `x in s` trả về `True` / `False`.

In [2]:
nums = [1, 1, 1000000000]
s = set(nums)

print(s)              # các số trùng bị gộp lại
print(len(nums), "phần tử trong list,", len(s), "phần tử trong set")
print(1 in s)         # True
print(5 in s)         # False

{1000000000, 1}
3 phần tử trong list, 2 phần tử trong set
True
False


## 2.2. `sorted(...)` và `.sort()` — sắp xếp

- `sorted(a)` tạo **list mới** đã sắp xếp, `a` giữ nguyên.
- `a.sort()` sắp xếp **ngay trên `a`** (in-place), không tạo list mới, trả về `None`.

Cả hai mất `O(n log n)`.

Đừng quên: `nums` trong bài **chưa chắc đã sắp xếp** (ví dụ 1 là `[3, 9, 7]`).

In [3]:
a = [3, 9, 7]

b = sorted(a)
print("sorted(a) =", b, "| a vẫn là", a)

a.sort()
print("sau a.sort(), a =", a)

print("a.sort() trả về:", [5, 1].sort())   # None -> đừng viết a = a.sort()

sorted(a) = [3, 7, 9] | a vẫn là [3, 9, 7]
sau a.sort(), a = [3, 7, 9]
a.sort() trả về: None


## 2.3. Lọc phần tử nằm trong `[lower, upper]`

Những số nằm ngoài đoạn không bao giờ ảnh hưởng tới kết quả, nên có thể lọc bỏ ngay:

```python
inside = [x for x in nums if lower <= x <= upper]
```

Python cho phép viết so sánh "kẹp giữa" `lower <= x <= upper`, tương đương `lower <= x and x <= upper`.

In [4]:
nums = [1, 1, 1000000000, 7, 3]
lower, upper = 5, 1000000000

inside = [x for x in nums if lower <= x <= upper]
print("Trong đoạn:", inside)
print("Đã bỏ trùng + sắp xếp:", sorted(set(inside)))

Trong đoạn: [1000000000, 7]
Đã bỏ trùng + sắp xếp: [7, 1000000000]


## 2.4. Duyệt các cặp phần tử kề nhau bằng `zip(a, a[1:])`

Ta thường cần xét từng cặp `(a[0], a[1]), (a[1], a[2]), ...`.

- `a[1:]` là list bỏ phần tử đầu.
- `zip(a, a[1:])` ghép phần tử thứ `i` của `a` với phần tử thứ `i` của `a[1:]`, tức là `a[i]` với `a[i+1]`.
- `zip` dừng khi list ngắn hơn hết phần tử, nên không bao giờ vượt chỉ số.

In [5]:
a = [0, 3, 7, 9, 13]

print("a      =", a)
print("a[1:]  =", a[1:])
for left, right in zip(a, a[1:]):
    print(f"cặp ({left}, {right}) -> chênh lệch {right - left}")

a      = [0, 3, 7, 9, 13]
a[1:]  = [3, 7, 9, 13]
cặp (0, 3) -> chênh lệch 3
cặp (3, 7) -> chênh lệch 4
cặp (7, 9) -> chênh lệch 2
cặp (9, 13) -> chênh lệch 4


# 3. Cách 1 — Brute Force: đi dọc trục số

Đây là cách tự nhiên nhất, giống hệt việc ta nhìn trục số ở mục 1.

Ý tưởng:

1. Bỏ `nums` vào một `set` để kiểm tra nhanh.
2. Duyệt `x` từ `lower` tới `upper`.
3. Dùng biến `start` để nhớ **đầu của đoạn thiếu đang mở**:
   - Gặp số **thiếu** mà chưa có đoạn mở → mở đoạn: `start = x`.
   - Gặp số **có mặt** mà đang có đoạn mở → đóng đoạn `[start, x - 1]`.
4. Hết vòng lặp mà đoạn vẫn mở → đóng đoạn `[start, upper]`.

Bảng trạng thái với ví dụ nhỏ `nums = [3]`, `lower = 1`, `upper = 5`:

| x | có mặt? | start trước | hành động | kết quả |
|---:|:---:|:---:|---|---|
| 1 | không | None | mở đoạn, start = 1 | [] |
| 2 | không | 1 | tiếp tục | [] |
| 3 | có | 1 | đóng [1, 2], start = None | [[1, 2]] |
| 4 | không | None | mở đoạn, start = 4 | [[1, 2]] |
| 5 | không | 4 | tiếp tục | [[1, 2]] |
| hết | | 4 | đóng [4, 5] | [[1, 2], [4, 5]] |

In [6]:
from typing import List


def missing_ranges_bruteforce(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    present = set(nums)
    result = []
    start = None                      # None nghĩa là "không có đoạn thiếu nào đang mở"

    for x in range(lower, upper + 1):
        if x not in present:
            if start is None:         # bắt đầu một đoạn thiếu mới
                start = x
        else:
            if start is not None:     # gặp số có mặt -> đóng đoạn đang mở
                result.append([start, x - 1])
                start = None

    if start is not None:             # đoạn thiếu kéo dài tới tận upper
        result.append([start, upper])

    return result


print(missing_ranges_bruteforce([3], 1, 5))
print(missing_ranges_bruteforce([3, 9, 7], 1, 12))
print(missing_ranges_bruteforce([2, 3, 5], 2, 3))

[[1, 2], [4, 5]]
[[1, 2], [4, 6], [8, 8], [10, 12]]
[]


## Độ phức tạp của Brute Force

Gọi `n = len(nums)` và `L = upper - lower + 1`.

- Tạo `set`: `O(n)`.
- Vòng lặp chạy đúng `L` lần.

```text
Time:  O(n + L)
Space: O(n)
```

Với ví dụ 2, `L` gần một tỷ. Cell dưới **không chạy** brute force (vì sẽ treo rất lâu), chỉ in ra số vòng lặp mà nó phải làm.

In [7]:
lower, upper = 5, 1_000_000_000
L = upper - lower + 1
print("Số vòng lặp brute force phải chạy:", f"{L:,}")
print("Trong khi nums chỉ có 3 phần tử!")

Số vòng lặp brute force phải chạy: 999,999,996
Trong khi nums chỉ có 3 phần tử!


Gần **1 tỷ** vòng lặp cho một mảng 3 phần tử — rõ ràng lãng phí.

Nút thắt: ta đang **đi qua từng số trên trục số**, kể cả những đoạn dài toàn số thiếu mà kết quả chỉ cần ghi bằng **hai đầu mút**.

---

# 4. Bước đột phá: chỉ "khoảng trống" giữa các số có mặt mới quan trọng

Hãy sắp xếp các số **có mặt trong đoạn** (đã bỏ trùng). Với ví dụ 1:

```text
lower = 1                                        upper = 12
  |                                                  |
  1   2  [3]  4   5   6  [7]  8  [9]  10  11  12
```

Các số thiếu luôn nằm **trong khoảng hở giữa hai số có mặt liên tiếp**:

| Hai số có mặt liên tiếp `a < b` | Khoảng hở | Đoạn thiếu |
|---|---|---|
| 3 và 7 | 4, 5, 6 | `[4, 6]` |
| 7 và 9 | 8 | `[8, 8]` |

Quy tắc tổng quát:

> Nếu `a` và `b` là hai số có mặt liên tiếp (sau khi sắp xếp, bỏ trùng) và `b - a >= 2`, thì đoạn thiếu giữa chúng là `[a + 1, b - 1]`.
>
> Nếu `b - a == 1` thì hai số sát nhau, không có gì thiếu.

Còn hai "mép": đoạn trước số có mặt nhỏ nhất (`1..2`) và đoạn sau số có mặt lớn nhất (`10..12`) — ta sẽ xử lý bằng một mẹo ở ngay mục sau.

Điều quan trọng nhất: **số khoảng hở không vượt quá `n + 1`**, nên công việc chỉ phụ thuộc `n`, không phụ thuộc `L`.

In [8]:
present_sorted = [3, 7, 9]

for a, b in zip(present_sorted, present_sorted[1:]):
    gap = b - a
    if gap >= 2:
        print(f"{a} -> {b}: chênh {gap} -> đoạn thiếu [{a + 1}, {b - 1}]")
    else:
        print(f"{a} -> {b}: chênh {gap} -> sát nhau, không thiếu")

3 -> 7: chênh 4 -> đoạn thiếu [4, 6]
7 -> 9: chênh 2 -> đoạn thiếu [8, 8]


## Mẹo lính canh (sentinel): biến hai mép thành khoảng hở bình thường

Vấn đề: đoạn thiếu ở đầu (`[1, 2]`) và ở cuối (`[10, 12]`) không nằm giữa hai số có mặt.

Mẹo: **giả vờ** có thêm hai số "có mặt" ngay sát bên ngoài đoạn:

- `lower - 1` đặt ở đầu,
- `upper + 1` đặt ở cuối.

```text
  [0]  1  2 [3] 4 5 6 [7] 8 [9] 10 11 12 [13]
   ^                                       ^
lower-1                                 upper+1
```

Bây giờ:

- Khoảng hở `0 -> 3` cho đoạn `[1, 2]`.
- Khoảng hở `9 -> 13` cho đoạn `[10, 12]`.

Vì `lower - 1` và `upper + 1` nằm **ngoài** đoạn `[lower, upper]`, chúng không bao giờ bị tính là số thiếu; chúng chỉ đóng vai "cột mốc". Các phần tử phụ như vậy gọi là **lính canh (sentinel)**: chúng giúp code không phải viết riêng các trường hợp ở mép.

In [9]:
lower, upper = 1, 12
present_sorted = [3, 7, 9]

vals = [lower - 1] + present_sorted + [upper + 1]
print("Có lính canh:", vals)

for a, b in zip(vals, vals[1:]):
    if b - a >= 2:
        print(f"khoảng hở {a} -> {b}: đoạn thiếu [{a + 1}, {b - 1}]")

Có lính canh: [0, 3, 7, 9, 13]
khoảng hở 0 -> 3: đoạn thiếu [1, 2]
khoảng hở 3 -> 7: đoạn thiếu [4, 6]
khoảng hở 7 -> 9: đoạn thiếu [8, 8]
khoảng hở 9 -> 13: đoạn thiếu [10, 12]


# 5. Cách 2 — Lọc + bỏ trùng + sắp xếp + lính canh

Ghép các mảnh lại:

1. Lọc các số nằm trong `[lower, upper]` và bỏ trùng bằng `set`.
2. Sắp xếp tăng dần.
3. Thêm lính canh: `[lower - 1] + ... + [upper + 1]`.
4. Duyệt từng cặp kề nhau `(a, b)`. Nếu `b - a >= 2` thì thêm `[a + 1, b - 1]`.

Vì sao phải **bỏ trùng**? Nếu giữ hai số `7, 7` thì cặp `(7, 7)` có chênh lệch 0 — không sai, nhưng thừa. Bỏ trùng giúp mỗi cặp kề nhau là hai số **thật sự khác nhau**.

Vì sao phải **lọc**? Ở ví dụ 2, nếu giữ số `1` (nhỏ hơn `lower = 5`) thì sau khi thêm lính canh `4`, ta có dãy `4, 1, ...` — không còn tăng dần, logic hỏng ngay.

In [10]:
def missing_ranges_sentinel(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    inside = sorted(set(x for x in nums if lower <= x <= upper))
    vals = [lower - 1] + inside + [upper + 1]

    result = []
    for a, b in zip(vals, vals[1:]):
        if b - a >= 2:
            result.append([a + 1, b - 1])
    return result


print(missing_ranges_sentinel([3, 9, 7], 1, 12))
print(missing_ranges_sentinel([1, 1, 1000000000], 5, 1000000000))
print(missing_ranges_sentinel([2, 3, 5], 2, 3))

[[1, 2], [4, 6], [8, 8], [10, 12]]
[[5, 999999999]]
[]


Ví dụ 2 giờ chạy **tức thì**, vì vòng lặp chỉ có vài cặp chứ không phải một tỷ số.

## Mô phỏng cách 2 với ví dụ 1

`nums = [3, 9, 7]`, `lower = 1`, `upper = 12` → `vals = [0, 3, 7, 9, 13]`

| Cặp (a, b) | b - a | Có thiếu? | Thêm vào kết quả |
|---|---:|:---:|---|
| (0, 3) | 3 | có | `[1, 2]` |
| (3, 7) | 4 | có | `[4, 6]` |
| (7, 9) | 2 | có | `[8, 8]` |
| (9, 13) | 4 | có | `[10, 12]` |

Hàm trace dưới đây in đúng bảng trên.

In [11]:
def trace_sentinel(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    inside = sorted(set(x for x in nums if lower <= x <= upper))
    vals = [lower - 1] + inside + [upper + 1]
    print("vals (đã có lính canh) =", vals)

    result = []
    for a, b in zip(vals, vals[1:]):
        if b - a >= 2:
            result.append([a + 1, b - 1])
            print(f"  cặp ({a}, {b}): chênh {b - a} -> thêm [{a + 1}, {b - 1}]")
        else:
            print(f"  cặp ({a}, {b}): chênh {b - a} -> sát nhau, bỏ qua")
    print("Kết quả:", result)
    return result


trace_sentinel([3, 9, 7], 1, 12)
print()
trace_sentinel([4, 4, 5, 6], 3, 6)

vals (đã có lính canh) = [0, 3, 7, 9, 13]
  cặp (0, 3): chênh 3 -> thêm [1, 2]
  cặp (3, 7): chênh 4 -> thêm [4, 6]
  cặp (7, 9): chênh 2 -> thêm [8, 8]
  cặp (9, 13): chênh 4 -> thêm [10, 12]
Kết quả: [[1, 2], [4, 6], [8, 8], [10, 12]]

vals (đã có lính canh) = [2, 4, 5, 6, 7]
  cặp (2, 4): chênh 2 -> thêm [3, 3]
  cặp (4, 5): chênh 1 -> sát nhau, bỏ qua
  cặp (5, 6): chênh 1 -> sát nhau, bỏ qua
  cặp (6, 7): chênh 1 -> sát nhau, bỏ qua
Kết quả: [[3, 3]]


[[3, 3]]

# 6. Cách 3 — Quét một lượt với biến `next_missing`

Cách 2 đã nhanh, nhưng phải tạo thêm `set` và list `vals`. Cách 3 chỉ cần **sắp xếp `nums` rồi quét một lượt** với một biến duy nhất.

## Ý nghĩa của biến `next_missing`

> `next_missing` = số nhỏ nhất **chưa được xử lý** trong đoạn `[lower, upper]`.
>
> Mọi số từ `lower` tới `next_missing - 1` đều đã được "giải quyết": hoặc đã nằm trong một đoạn thiếu đã ghi, hoặc là số có mặt.

Ban đầu chưa xử lý gì: `next_missing = lower`.

Duyệt từng `x` trong `nums` đã sắp xếp:

1. Nếu `x` nằm ngoài `[lower, upper]` → bỏ qua.
2. Nếu `x > next_missing` → các số `next_missing, ..., x - 1` đều thiếu → ghi đoạn `[next_missing, x - 1]`.
3. Số `x` có mặt, vậy số chưa xử lý tiếp theo là `x + 1` → `next_missing = max(next_missing, x + 1)`.

Hết vòng lặp, nếu `next_missing <= upper` thì đoạn `[next_missing, upper]` còn thiếu.

## Số trùng lặp được xử lý thế nào?

Với `nums = [3, 3]`: gặp `3` đầu tiên → `next_missing = 4`. Gặp `3` thứ hai: `3 > 4` sai nên không ghi gì, và `max(4, 3 + 1) = 4` giữ nguyên. Không cần `set`!

Dùng `max` là cách viết phòng thủ: dù vì lý do nào đó `x + 1` nhỏ hơn giá trị hiện tại, `next_missing` cũng **không bao giờ lùi lại**.

In [12]:
def missing_ranges_sweep(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    result = []
    next_missing = lower

    for x in sorted(nums):
        if x < lower or x > upper:
            continue                                  # ngoài đoạn: không liên quan
        if x > next_missing:
            result.append([next_missing, x - 1])      # khoảng hở trước x
        next_missing = max(next_missing, x + 1)       # x có mặt -> nhảy qua x

    if next_missing <= upper:
        result.append([next_missing, upper])          # phần đuôi còn thiếu
    return result


print(missing_ranges_sweep([3, 9, 7], 1, 12))
print(missing_ranges_sweep([1, 1, 1000000000], 5, 1000000000))
print(missing_ranges_sweep([2, 3, 5], 2, 3))

[[1, 2], [4, 6], [8, 8], [10, 12]]
[[5, 999999999]]
[]


## Mô phỏng cách 3 — ví dụ có đủ các nhánh đặc biệt

Chọn một input "khó" để thấy mọi nhánh: có số ngoài đoạn, có số trùng, có số sát nhau.

```python
nums = [20, 5, 3, 3, 8, 1, 4], lower = 2, upper = 9
```

Sau khi sắp xếp: `[1, 3, 3, 4, 5, 8, 20]`.

| x | trong đoạn? | next_missing trước | x > next_missing? | ghi đoạn | next_missing sau |
|---:|:---:|---:|:---:|---|---:|
| 1 | không | 2 | — | — | 2 |
| 3 | có | 2 | có | `[2, 2]` | 4 |
| 3 | có | 4 | không | — | 4 |
| 4 | có | 4 | không | — | 5 |
| 5 | có | 5 | không | — | 6 |
| 8 | có | 6 | có | `[6, 7]` | 9 |
| 20 | không | 9 | — | — | 9 |
| hết | | 9 | `9 <= 9` | `[9, 9]` | |

Kết quả: `[[2, 2], [6, 7], [9, 9]]`.

In [13]:
def trace_sweep(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    result = []
    next_missing = lower
    print(f"sorted(nums) = {sorted(nums)}, lower = {lower}, upper = {upper}")

    for x in sorted(nums):
        if x < lower or x > upper:
            print(f"x = {x:>2}: ngoài đoạn -> bỏ qua")
            continue
        before = next_missing
        if x > next_missing:
            result.append([next_missing, x - 1])
            action = f"ghi [{next_missing}, {x - 1}]"
        else:
            action = "không có khoảng hở"
        next_missing = max(next_missing, x + 1)
        print(f"x = {x:>2}: next_missing {before} -> {next_missing} | {action}")

    if next_missing <= upper:
        result.append([next_missing, upper])
        print(f"Hết mảng: next_missing = {next_missing} <= upper -> ghi [{next_missing}, {upper}]")
    print("Kết quả:", result)
    return result


trace_sweep([20, 5, 3, 3, 8, 1, 4], 2, 9)

sorted(nums) = [1, 3, 3, 4, 5, 8, 20], lower = 2, upper = 9
x =  1: ngoài đoạn -> bỏ qua
x =  3: next_missing 2 -> 4 | ghi [2, 2]
x =  3: next_missing 4 -> 4 | không có khoảng hở
x =  4: next_missing 4 -> 5 | không có khoảng hở
x =  5: next_missing 5 -> 6 | không có khoảng hở
x =  8: next_missing 6 -> 9 | ghi [6, 7]
x = 20: ngoài đoạn -> bỏ qua
Hết mảng: next_missing = 9 <= upper -> ghi [9, 9]
Kết quả: [[2, 2], [6, 7], [9, 9]]


[[2, 2], [6, 7], [9, 9]]

# 7. Phiên bản đúng format LeetCode

Ta chọn **cách 3** để nộp: ngắn, chỉ một vòng lặp, không cần tạo `set`.

Lưu ý:

- Nếu chạy ngoài LeetCode, cần `from typing import List`.
- Ở đây dùng `sorted(nums)` để **không làm thay đổi** mảng đầu vào. Nếu muốn tiết kiệm bộ nhớ hơn có thể dùng `nums.sort()` (sắp xếp tại chỗ), LeetCode vẫn chấp nhận.

In [14]:
from typing import List


class Solution:
    def findDisappearedNumbers(self, nums: List[int], lower: int, upper: int) -> List[List[int]]:
        result = []
        next_missing = lower

        for x in sorted(nums):
            if x < lower or x > upper:
                continue
            if x > next_missing:
                result.append([next_missing, x - 1])
            next_missing = max(next_missing, x + 1)

        if next_missing <= upper:
            result.append([next_missing, upper])
        return result

In [15]:
sol = Solution()

tests = [
    # (nums, lower, upper, expected, mô tả)
    ([3, 9, 7], 1, 12, [[1, 2], [4, 6], [8, 8], [10, 12]], "ví dụ 1"),
    ([1, 1, 1000000000], 5, 1000000000, [[5, 999999999]], "ví dụ 2: số ngoài đoạn + upper lớn"),
    ([2, 3, 5], 2, 3, [], "ví dụ 3: không thiếu gì"),
    ([100, 200], 1, 10, [[1, 10]], "mọi số đều ngoài đoạn"),
    ([5], 5, 5, [], "lower == upper và có mặt"),
    ([4], 5, 5, [[5, 5]], "lower == upper và thiếu"),
    ([7, 7, 7, 7], 6, 8, [[6, 6], [8, 8]], "toàn số trùng"),
    ([1, 2, 3, 4], 1, 4, [], "phủ kín đoạn"),
    ([10, 1], 1, 10, [[2, 9]], "chỉ có hai mép"),
]

for nums, lower, upper, expected, name in tests:
    result = sol.findDisappearedNumbers(nums, lower, upper)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:38} -> {result}")

PASS | ví dụ 1                                -> [[1, 2], [4, 6], [8, 8], [10, 12]]
PASS | ví dụ 2: số ngoài đoạn + upper lớn     -> [[5, 999999999]]
PASS | ví dụ 3: không thiếu gì                -> []
PASS | mọi số đều ngoài đoạn                  -> [[1, 10]]
PASS | lower == upper và có mặt               -> []
PASS | lower == upper và thiếu                -> [[5, 5]]
PASS | toàn số trùng                          -> [[6, 6], [8, 8]]
PASS | phủ kín đoạn                           -> []
PASS | chỉ có hai mép                         -> [[2, 9]]


# 8. Vì sao thuật toán đúng?

Ta chứng minh cho cách 3 bằng **bất biến (invariant)** — một mệnh đề luôn đúng sau mỗi vòng lặp:

> **Bất biến:** sau khi xử lý xong các số `<= x` (trong mảng đã sắp xếp), thì:
> 1. mọi số thiếu nhỏ hơn `next_missing` đã được ghi vào `result`, theo đúng thứ tự, đã gộp tối đa;
> 2. `next_missing` có mặt trong mảng **hoặc** chưa được xét tới (chưa biết có mặt hay không).

Kiểm tra từng bước:

- **Ban đầu:** `next_missing = lower`, chưa có số nào nhỏ hơn `lower` cần quan tâm → đúng.
- **Gặp `x > next_missing`:** vì mảng đã sắp xếp, mọi số có mặt trước `x` đều `< next_missing`. Do đó các số `next_missing, ..., x - 1` chắc chắn **không** có trong mảng → đây là một đoạn thiếu. Nó **không thể gộp** với đoạn trước (vì `next_missing - 1` có mặt hoặc là `lower - 1`), cũng không thể kéo dài qua `x` (vì `x` có mặt) → đoạn được gộp tối đa.
- **Gặp `x <= next_missing`:** `x` đã nằm trong vùng đã xử lý hoặc chính là `next_missing` → không có khoảng hở mới.
- **Kết thúc:** mọi số có mặt đều `< next_missing`, nên `next_missing..upper` (nếu không rỗng) là đoạn thiếu cuối cùng.

Các đoạn được ghi theo thứ tự `next_missing` tăng dần, nên kết quả tự động **sắp xếp theo `start`**.

---

## Trực giác ngắn gọn

> Đi trên trục số nhưng **nhảy cóc** từ số có mặt này sang số có mặt kế tiếp. Mỗi cú nhảy qua một khoảng trống chính là một đoạn thiếu.

# 9. Độ phức tạp

Gọi `n = len(nums)`, `L = upper - lower + 1`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Brute Force | đi qua từng số trên trục | `O(n + L)` | `O(n)` |
| 2. Lính canh | lọc + `set` + sort + xét cặp kề | `O(n log n)` | `O(n)` |
| 3. Quét `next_missing` | sort + một vòng lặp | `O(n log n)` | `O(n)` nếu dùng `sorted`, `O(1)` thêm nếu `nums.sort()` |

(Không tính bộ nhớ của list kết quả — kết quả có tối đa `n + 1` đoạn.)

Phần tốn thời gian nhất của cách 2 và 3 là **sắp xếp**. Vòng lặp sau đó chỉ `O(n)`.

Vì `n <= 10^5`, `n log n` chỉ khoảng `1.7 triệu` phép toán → rất nhanh. Trong khi `L` có thể là `10^9`.

# 10. Đối chiếu ba cách trên dữ liệu ngẫu nhiên

Khi có một lời giải chậm nhưng chắc chắn đúng (brute force), ta dùng nó để **kiểm tra chéo** lời giải nhanh trên hàng nghìn input nhỏ ngẫu nhiên. Đây là thói quen rất tốt khi luyện thuật toán.

In [16]:
import random

random.seed(2026)

for _ in range(2000):
    lower = random.randint(1, 15)
    upper = random.randint(lower, 25)
    nums = [random.randint(1, 30) for _ in range(random.randint(1, 10))]

    expected = missing_ranges_bruteforce(nums, lower, upper)
    assert missing_ranges_sentinel(nums, lower, upper) == expected, (nums, lower, upper)
    assert missing_ranges_sweep(nums, lower, upper) == expected, (nums, lower, upper)
    assert sol.findDisappearedNumbers(nums, lower, upper) == expected, (nums, lower, upper)

print("2000 bộ test ngẫu nhiên: cả 3 cách cho cùng kết quả!")

2000 bộ test ngẫu nhiên: cả 3 cách cho cùng kết quả!


# 11. Những lỗi dễ mắc

## Lỗi 1 — Quên bỏ qua các số nằm ngoài `[lower, upper]`

Code sai (không lọc):

```python
for x in sorted(nums):
    if x > next_missing:
        result.append([next_missing, x - 1])
    next_missing = max(next_missing, x + 1)
```

Với ví dụ 2 (`nums = [1, 1, 10^9]`, `lower = 5`) thì không sao, nhưng nếu có số **lớn hơn `upper`**, code sai sẽ sinh đoạn vượt quá `upper`. Cell dưới cho thấy điều đó.

In [17]:
def wrong_no_filter(nums, lower, upper):
    result = []
    next_missing = lower
    for x in sorted(nums):
        if x > next_missing:
            result.append([next_missing, x - 1])
        next_missing = max(next_missing, x + 1)
    if next_missing <= upper:
        result.append([next_missing, upper])
    return result


print("Sai :", wrong_no_filter([3, 50], 1, 10))        # có đoạn [4, 49] vượt quá upper = 10
print("Đúng:", missing_ranges_sweep([3, 50], 1, 10))

Sai : [[1, 2], [4, 49]]
Đúng: [[1, 2], [4, 10]]


## Lỗi 2 — Dùng `!=` thay cho `>` khi kiểm tra khoảng hở

Code sai:

```python
if x != next_missing:
    result.append([next_missing, x - 1])
```

Khi có số trùng, ví dụ `nums = [3, 3]`, `lower = 1`: gặp `3` lần hai thì `next_missing = 4`, `3 != 4` đúng → ghi đoạn `[4, 2]` — một đoạn **ngược**, vô nghĩa.

Phải dùng `x > next_missing`: chỉ khi `x` nằm **phía sau** vùng đã xử lý mới có khoảng hở.

In [18]:
def wrong_not_equal(nums, lower, upper):
    result = []
    next_missing = lower
    for x in sorted(nums):
        if x < lower or x > upper:
            continue
        if x != next_missing:                     # SAI
            result.append([next_missing, x - 1])
        next_missing = max(next_missing, x + 1)
    if next_missing <= upper:
        result.append([next_missing, upper])
    return result


print("Sai :", wrong_not_equal([3, 3], 1, 5))
print("Đúng:", missing_ranges_sweep([3, 3], 1, 5))

Sai : [[1, 2], [4, 2], [4, 5]]
Đúng: [[1, 2], [4, 5]]


## Lỗi 3 — Lệch một đơn vị (off-by-one) ở hai đầu đoạn

Giữa hai số có mặt `a < b`, đoạn thiếu là `[a + 1, b - 1]`, **không phải** `[a, b]`.

Mẹo kiểm tra nhanh: thử với `a = 7`, `b = 9` → đoạn thiếu phải là `[8, 8]` (một số). Nếu công thức cho ra `[7, 9]` hoặc `[8, 9]` là sai.

---

## Lỗi 4 — Quên phần đuôi sau số có mặt cuối cùng

Nếu thiếu đoạn code:

```python
if next_missing <= upper:
    result.append([next_missing, upper])
```

thì ví dụ 1 sẽ mất đoạn `[10, 12]`. Ngược lại, nếu viết `<` thay cho `<=`, sẽ mất đoạn chỉ gồm đúng số `upper` (ví dụ `[9, 9]` trong mô phỏng ở mục 6).

---

## Lỗi 5 — Dùng Brute Force với `range(lower, upper + 1)`

Code đúng logic nhưng với `upper - lower` cỡ `10^9` sẽ bị **Time Limit Exceeded**. Luôn nhìn constraints trước khi chọn cách.

---

## Lỗi 6 — Viết `nums = nums.sort()`

`.sort()` trả về `None`, nên sau dòng này `nums` thành `None` và vòng `for` sẽ báo lỗi. Hoặc viết `nums.sort()` (không gán), hoặc `nums = sorted(nums)`.

# 12. Mẹo nhận diện kỹ thuật "khoảng trống giữa các điểm"

Hãy nghĩ tới việc **sắp xếp rồi xét khoảng hở giữa hai phần tử liên tiếp** khi thấy:

1. **Miền giá trị rất lớn** (`10^9`) nhưng **số phần tử nhỏ** (`10^5`) → không được duyệt theo giá trị, phải duyệt theo phần tử.
2. Đề hỏi về **"đoạn"**, **"khoảng"**, **"range"**, **"liên tiếp"**, **"bị thiếu"**, **"không bị phủ"**.
3. Kết quả là các đoạn `[start, end]` — chỉ cần hai đầu mút, không cần liệt kê từng số.

Hai mẹo đi kèm:

- **Lính canh (sentinel):** thêm phần tử giả ở hai đầu (`lower - 1`, `upper + 1`) để không phải viết code riêng cho mép.
- **Biến "điểm chưa xử lý"** (`next_missing`): một con trỏ chạy trên trục số, chỉ tiến không lùi.

Các bài cùng họ: *Missing Ranges* (LeetCode 163), *Summary Ranges* (228), *Merge Intervals* (56), *Insert Interval* (57).

# 13. Biến thể

## Biến thể 1 — Chỉ đếm **tổng số lượng** số bị thiếu

Không cần liệt kê đoạn, chỉ cần một con số. Mẹo: số thiếu = `L` trừ đi **số giá trị khác nhau có mặt trong đoạn**.

```text
missing_count = (upper - lower + 1) - len(set of x in nums with lower <= x <= upper)
```

In [19]:
def count_missing(nums: List[int], lower: int, upper: int) -> int:
    distinct_inside = {x for x in nums if lower <= x <= upper}
    return (upper - lower + 1) - len(distinct_inside)


print(count_missing([3, 9, 7], 1, 12))                      # 12 - 3 = 9
print(count_missing([1, 1, 1000000000], 5, 1000000000))      # 999999995
# kiểm tra lại bằng tổng độ dài các đoạn thiếu
ranges = missing_ranges_sweep([3, 9, 7], 1, 12)
print(sum(end - start + 1 for start, end in ranges))

9
999999995
9


## Biến thể 2 — Đoạn **có mặt** thay vì đoạn thiếu (Summary Ranges)

Đảo vai trò: gộp các số **có mặt** liên tiếp trong `[lower, upper]` thành đoạn.

Ví dụ `nums = [3, 9, 7, 8, 4]`, `lower = 1`, `upper = 12` → các số có mặt `3, 4, 7, 8, 9` → `[[3, 4], [7, 9]]`.

Ý tưởng: sắp xếp, bỏ trùng; một đoạn có mặt **kết thúc** khi phần tử tiếp theo không bằng phần tử hiện tại `+ 1`.

In [20]:
def present_ranges(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    vals = sorted({x for x in nums if lower <= x <= upper})
    result = []
    i = 0
    while i < len(vals):
        j = i
        while j + 1 < len(vals) and vals[j + 1] == vals[j] + 1:
            j += 1
        result.append([vals[i], vals[j]])
        i = j + 1
    return result


print(present_ranges([3, 9, 7, 8, 4], 1, 12))
print(present_ranges([5, 5, 6], 1, 10))

[[3, 4], [7, 9]]
[[5, 6]]


Nhận xét: đoạn thiếu và đoạn có mặt **xen kẽ nhau** trên trục số. Hiểu một bài là hiểu luôn bài kia.

---

# 14. Template tư duy

```python
# Bài toán dạng: miền giá trị lớn, cần tìm các đoạn thiếu / khoảng hở
result = []
next_missing = lower                # điểm nhỏ nhất chưa xử lý

for x in sorted(points):
    if x < lower or x > upper:      # 1. bỏ điểm ngoài miền
        continue
    if x > next_missing:            # 2. có khoảng hở trước x
        result.append([next_missing, x - 1])
    next_missing = max(next_missing, x + 1)   # 3. nhảy qua x

if next_missing <= upper:           # 4. đừng quên phần đuôi
    result.append([next_missing, upper])
```

Phiên bản lính canh:

```python
vals = [lower - 1] + sorted(set(inside_points)) + [upper + 1]
for a, b in zip(vals, vals[1:]):
    if b - a >= 2:
        result.append([a + 1, b - 1])
```

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Hãy tự tìm kết quả:

```python
nums = [10, 4, 4, 7], lower = 3, upper = 11
```

Gợi ý quy trình: lọc → bỏ trùng → sắp xếp → thêm lính canh → xét từng cặp.

Sau khi tính xong, dùng cell dưới để kiểm tra.

In [21]:
# Kiểm tra Bài 1 sau khi đã tự tính bằng tay
trace_sentinel([10, 4, 4, 7], 3, 11);

vals (đã có lính canh) = [2, 4, 7, 10, 12]
  cặp (2, 4): chênh 2 -> thêm [3, 3]
  cặp (4, 7): chênh 3 -> thêm [5, 6]
  cặp (7, 10): chênh 3 -> thêm [8, 9]
  cặp (10, 12): chênh 2 -> thêm [11, 11]
Kết quả: [[3, 3], [5, 6], [8, 9], [11, 11]]


## Bài 2 — Dự đoán output

Đoán xem đoạn code sau in ra gì, rồi mới chạy cell:

```python
vals = [0, 1, 2, 6, 7, 11]
print([b - a for a, b in zip(vals, vals[1:])])
print([[a + 1, b - 1] for a, b in zip(vals, vals[1:]) if b - a >= 2])
```

In [22]:
vals = [0, 1, 2, 6, 7, 11]
print([b - a for a, b in zip(vals, vals[1:])])
print([[a + 1, b - 1] for a, b in zip(vals, vals[1:]) if b - a >= 2])

[1, 1, 4, 1, 4]
[[3, 5], [8, 10]]


## Bài 3 — Hoàn thiện cách lính canh

Điền vào các chỗ `TODO`.

In [23]:
def exercise_3_sentinel(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    # TODO 1: lọc các x nằm trong [lower, upper], bỏ trùng, sắp xếp -> inside
    # TODO 2: tạo vals = [lower - 1] + inside + [upper + 1]
    # TODO 3: duyệt từng cặp (a, b) kề nhau; nếu b - a >= 2 thì thêm [a + 1, b - 1]
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# Bỏ comment để kiểm tra sau khi làm xong:
# assert exercise_3_sentinel([3, 9, 7], 1, 12) == [[1, 2], [4, 6], [8, 8], [10, 12]]
# assert exercise_3_sentinel([2, 3, 5], 2, 3) == []

## Bài 4 — Tự viết lại cách quét `next_missing`

Không nhìn code phía trên, tự viết lại hàm. Yêu cầu:

- Không dùng `set`.
- Xử lý đúng số trùng và số ngoài đoạn.

In [24]:
def exercise_4_sweep(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    # TODO: tự viết
    raise NotImplementedError


# assert exercise_4_sweep([20, 5, 3, 3, 8, 1, 4], 2, 9) == [[2, 2], [6, 7], [9, 9]]
# assert exercise_4_sweep([1, 1, 1000000000], 5, 1000000000) == [[5, 999999999]]

## Bài 5 — Biến thể: đoạn thiếu **dài nhất**

Trả về độ dài của đoạn thiếu dài nhất trong `[lower, upper]` (trả về `0` nếu không thiếu số nào).

Ví dụ:

```python
longest_missing_run([3, 9, 7], 1, 12)   -> 3   # đoạn [4, 6] hoặc [10, 12]
longest_missing_run([2, 3], 2, 3)       -> 0
```

Yêu cầu: chạy nhanh kể cả khi `upper = 10^9`.

In [25]:
def exercise_5_longest_missing_run(nums: List[int], lower: int, upper: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_longest_missing_run([3, 9, 7], 1, 12) == 3
# assert exercise_5_longest_missing_run([2, 3], 2, 3) == 0
# assert exercise_5_longest_missing_run([1], 1, 10**9) == 10**9 - 1

## Bài 6 — Mini challenge: số thiếu thứ `k`

Trả về **số thiếu thứ `k`** (đếm từ 1, theo thứ tự tăng dần) trong `[lower, upper]`. Nếu không có đủ `k` số thiếu, trả về `-1`.

Ví dụ với `nums = [3, 9, 7]`, `lower = 1`, `upper = 12`, các số thiếu theo thứ tự là `1, 2, 4, 5, 6, 8, 10, 11, 12`:

```python
kth_missing([3, 9, 7], 1, 12, 1)   -> 1
kth_missing([3, 9, 7], 1, 12, 3)   -> 4
kth_missing([3, 9, 7], 1, 12, 9)   -> 12
kth_missing([3, 9, 7], 1, 12, 10)  -> -1
```

Yêu cầu: không được liệt kê từng số (vì `k` có thể tới `10^9`).

In [26]:
def exercise_6_kth_missing(nums: List[int], lower: int, upper: int, k: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_6_kth_missing([3, 9, 7], 1, 12, 1) == 1
# assert exercise_6_kth_missing([3, 9, 7], 1, 12, 3) == 4
# assert exercise_6_kth_missing([3, 9, 7], 1, 12, 9) == 12
# assert exercise_6_kth_missing([3, 9, 7], 1, 12, 10) == -1

## Gợi ý (mở từng mức, đừng đọc hết một lúc)

**Bài 3**

- Mức 1: kết quả mong muốn chính là "khoảng hở" giữa các số liên tiếp.
- Mức 2: `inside = sorted(set(x for x in nums if lower <= x <= upper))`.
- Mức 3: `for a, b in zip(vals, vals[1:]): if b - a >= 2: result.append([a + 1, b - 1])`.

**Bài 4**

- Mức 1: cần một biến nhớ "số nhỏ nhất chưa xử lý".
- Mức 2: so sánh `x > next_missing` (không phải `!=`).
- Mức 3: cập nhật `next_missing = max(next_missing, x + 1)` và đừng quên phần đuôi.

**Bài 5**

- Mức 1: đoạn thiếu dài nhất là một trong các đoạn của bài gốc.
- Mức 2: đoạn `[s, e]` có độ dài `e - s + 1`.
- Mức 3: `max((e - s + 1 for s, e in ranges), default=0)`.

**Bài 6**

- Mức 1: duyệt từng **đoạn** thiếu thay vì từng số. Mỗi đoạn chứa `e - s + 1` số.
- Mức 2: nếu `k` lớn hơn độ dài đoạn hiện tại thì trừ `k` đi độ dài đó và sang đoạn tiếp theo.
- Mức 3: nếu `k <= e - s + 1` thì đáp án là `s + k - 1`.

# 16. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

Lọc trong `[3, 11]`: `10, 4, 4, 7` → bỏ trùng, sắp xếp: `[4, 7, 10]` → lính canh: `[2, 4, 7, 10, 12]`.

| Cặp | Chênh | Đoạn |
|---|---:|---|
| (2, 4) | 2 | `[3, 3]` |
| (4, 7) | 3 | `[5, 6]` |
| (7, 10) | 3 | `[8, 9]` |
| (10, 12) | 2 | `[11, 11]` |

Kết quả: `[[3, 3], [5, 6], [8, 9], [11, 11]]`.

## Đáp án Bài 2

```text
[1, 1, 4, 1, 4]
[[3, 5], [8, 10]]
```

In [27]:
# Đáp án Bài 3
def answer_3_sentinel(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    inside = sorted(set(x for x in nums if lower <= x <= upper))
    vals = [lower - 1] + inside + [upper + 1]
    result = []
    for a, b in zip(vals, vals[1:]):
        if b - a >= 2:
            result.append([a + 1, b - 1])
    return result


assert answer_3_sentinel([3, 9, 7], 1, 12) == [[1, 2], [4, 6], [8, 8], [10, 12]]
assert answer_3_sentinel([2, 3, 5], 2, 3) == []
assert answer_3_sentinel([10, 4, 4, 7], 3, 11) == [[3, 3], [5, 6], [8, 9], [11, 11]]
print("Bài 3: PASS")

Bài 3: PASS


In [28]:
# Đáp án Bài 4
def answer_4_sweep(nums: List[int], lower: int, upper: int) -> List[List[int]]:
    result = []
    next_missing = lower
    for x in sorted(nums):
        if x < lower or x > upper:
            continue
        if x > next_missing:
            result.append([next_missing, x - 1])
        next_missing = max(next_missing, x + 1)
    if next_missing <= upper:
        result.append([next_missing, upper])
    return result


assert answer_4_sweep([20, 5, 3, 3, 8, 1, 4], 2, 9) == [[2, 2], [6, 7], [9, 9]]
assert answer_4_sweep([1, 1, 1000000000], 5, 1000000000) == [[5, 999999999]]
print("Bài 4: PASS")

Bài 4: PASS


In [29]:
# Đáp án Bài 5
def answer_5_longest_missing_run(nums: List[int], lower: int, upper: int) -> int:
    ranges = answer_4_sweep(nums, lower, upper)
    return max((end - start + 1 for start, end in ranges), default=0)


assert answer_5_longest_missing_run([3, 9, 7], 1, 12) == 3
assert answer_5_longest_missing_run([2, 3], 2, 3) == 0
assert answer_5_longest_missing_run([1], 1, 10**9) == 10**9 - 1
print("Bài 5: PASS")

Bài 5: PASS


In [30]:
# Đáp án Bài 6
def answer_6_kth_missing(nums: List[int], lower: int, upper: int, k: int) -> int:
    for start, end in answer_4_sweep(nums, lower, upper):
        length = end - start + 1
        if k <= length:
            return start + k - 1
        k -= length
    return -1


assert answer_6_kth_missing([3, 9, 7], 1, 12, 1) == 1
assert answer_6_kth_missing([3, 9, 7], 1, 12, 3) == 4
assert answer_6_kth_missing([3, 9, 7], 1, 12, 9) == 12
assert answer_6_kth_missing([3, 9, 7], 1, 12, 10) == -1
assert answer_6_kth_missing([1, 1, 1000000000], 5, 1000000000, 999999995) == 999999999
print("Bài 6: PASS")

Bài 6: PASS


# 17. Tổng kết

Bài **Find All Numbers Disappeared in an Array III** dạy một tư duy rất quan trọng:

> Khi miền giá trị khổng lồ nhưng số điểm ít, **đừng đi theo giá trị — hãy đi theo điểm**. Sắp xếp các điểm và chỉ xét **khoảng trống giữa hai điểm liên tiếp**.

## Công thức nhớ nhanh

```text
Giữa hai số có mặt a < b (đã sort, bỏ trùng):
    nếu b - a >= 2  ->  đoạn thiếu [a + 1, b - 1]
Thêm lính canh lower - 1 và upper + 1 để xử lý hai mép.
```

## Phiên bản nên nhớ để nộp

```python
class Solution:
    def findDisappearedNumbers(self, nums: List[int], lower: int, upper: int) -> List[List[int]]:
        result = []
        next_missing = lower
        for x in sorted(nums):
            if x < lower or x > upper:
                continue
            if x > next_missing:
                result.append([next_missing, x - 1])
            next_missing = max(next_missing, x + 1)
        if next_missing <= upper:
            result.append([next_missing, upper])
        return result
```

## Độ phức tạp

```text
Time:  O(n log n)   (do sắp xếp)
Space: O(n)         (sorted tạo bản sao; O(1) thêm nếu dùng nums.sort())
```

# 18. Checklist trước khi submit

- [ ] Mình có **sắp xếp** `nums` chưa? (đề không hứa mảng đã sắp xếp)
- [ ] Mình có **bỏ qua số nằm ngoài** `[lower, upper]` chưa?
- [ ] Số **trùng lặp** có làm sinh đoạn ngược như `[4, 2]` không? (dùng `>` chứ không dùng `!=`)
- [ ] Đoạn giữa hai số có mặt là `[a + 1, b - 1]`, không lệch một đơn vị?
- [ ] Đã xử lý **phần đầu** (trước số có mặt nhỏ nhất) và **phần đuôi** (sau số có mặt lớn nhất) chưa?
- [ ] Trường hợp `lower == upper` chạy đúng chưa?
- [ ] Có chỗ nào lỡ duyệt `range(lower, upper + 1)` không? (sẽ TLE khi `upper = 10^9`)
- [ ] Không viết `nums = nums.sort()`?